In [1]:
import sys
sys.path.append("../")
import os
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
import requests
from src.chunking import chunk_text
from src.loaders.txt_loader import load_txt
from src.loaders.pdf_loader import load_pdf
from src.loaders.document_loader import load_documents
from src.pipeline.build_docs import build_docs
from src.vector_store.faiss_store import build_index

d:\Program\anaconda3\envs\rag\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ["HF_HOME"] = r"G:\huggingface_cache"

In [3]:
# 1. embedding模型（轻量）
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1398.88it/s]


In [4]:
loaded_docs = load_documents("../data")
docs = build_docs(loaded_docs)

In [5]:
len(docs)
docs[70:]

[{'chunk_id': 70,
  'source': 'tech_history.txt',
  'page': None,
  'text': 'puting remains important in areas such as gaming, privacy-sensitive applications, and real-time processing.'},
 {'chunk_id': 71,
  'source': 'urban_transport.txt',
  'page': None,
  'text': 'Urban transportation systems influence how cities grow and how people experience daily life. Efficient movement of people can improve productivity, reduce pollution, and increase accessibility.\n\nTradi'},
 {'chunk_id': 72,
  'source': 'urban_transport.txt',
  'page': None,
  'text': 'duce pollution, and increase accessibility.\n\nTraditional transportation networks relied heavily on private vehicles. While cars provide flexibility, large-scale dependence often creates congestion and'},
 {'chunk_id': 73,
  'source': 'urban_transport.txt',
  'page': None,
  'text': 'arge-scale dependence often creates congestion and infrastructure pressure. Many cities have therefore invested in public transportation systems.\n\nRail networ

In [6]:
index = build_index(model, docs)

In [7]:
def retrieve(query, k=2):
    q_emb = model.encode([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    distances, indices = index.search(q_emb, k)
    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            "chunk_id": docs[idx]["chunk_id"],
            "source": docs[idx]["source"],
            "score": float(distances[0][rank]),
            "page": docs[idx]["page"],
            "text": docs[idx]["text"]
        })
    return results

def ask_llm(context, question):
    prompt = f"""
You are a helpful assistant.

Context:
{context}

Question:
{question}
"""

    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:7b",
            "prompt": prompt,
            "stream": False
        }
    )

    return res.json()["response"]

In [8]:
if __name__ == "__main__":
    query = "What users prefer in AI systems?"

    retrieved = retrieve(query)
    context = "\n".join(r["text"] for r in retrieved)

    answer = ask_llm(context, query)

    print("\n=== Retrieved ===")

    for r in retrieved:
        print(
            f"""
            Chunk {r['chunk_id']}
            source {r["source"]}
            Page {r["page"]}
            score:{r['score']:.3f}
            {r['text']}
            """
        )


=== Retrieved ===

            Chunk 29
            source sample.pdf
            Page 2
            score:0.642
            rove retrieval. 
 
Human Interaction 
Successful AI systems depend not only on algorithms but also on user experience. 
Users often prefer systems that: 
• respond quickly, 
• explain results clearly,
            

            Chunk 17
            source sample.pdf
            Page 1
            score:0.594
            isting software entirely, many AI systems act as assistants that extend human 
capabilities. They summarize information, generate content, and aut omate repetitive 
processes. 
As these tools improve,
            


In [ ]:
print("\n=== Answer ===")
print(answer)
print("\n=== Sources ===")
seen = set() #去重
for r in retrieved:
    source = (
        f"{r['source']}"
        if r["page"] is None
        else
        f"{r['source']} Page {r['page']}"
    )

    if source not in seen:
        print(source)
        seen.add(source)


=== Answer ===
Users generally prefer AI systems that:

1. **Respond Quickly:** Users appreciate systems that provide fast responses to their queries or requests. Quick response times enhance the overall user experience and can make interactions feel more natural.

2. **Explain Results Clearly:** For many users, especially those who are not deeply familiar with AI technologies, clear explanations of how results were generated or why certain outcomes occurred can significantly improve trust and satisfaction. This transparency helps users understand and validate the AI's decisions or recommendations.

3. **Extend Human Capabilities:** Many AI systems act as helpful assistants that augment human capabilities rather than replacing them entirely. By summarizing information, generating content, or automating repetitive processes, these tools can save time and effort, making work more efficient.

4. **Provide Accurate and Reliable Information:** Users expect the results from AI to be accurat